# Modelado - Enfoque A: Validación Estática (Hold-out)
En esta notebook utilizaremos la estrategia clásica de validación estática, donde hemos separado un 70% para entrenamiento y un 15% para validación (dejando el 15% final para test).

In [ ]:
import math
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Modelos Baseline
from sklearn.dummy import DummyClassifier

# Modelos Lineales
from sklearn.linear_model import LogisticRegression

# Árboles y Ensambles
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Análisis Discriminante
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Naive Bayes
from sklearn.naive_bayes import GaussianNB

# Vecinos más cercanos (k-NN)
from sklearn.neighbors import KNeighborsClassifier

# Redes Neuronales
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score, precision_score, f1_score, recall_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


### 1. Carga de Datos

In [ ]:
base_path = '../data/model_input/cv/'

X_train_full = pd.read_csv(base_path + 'X_train_full.csv')
y_train_full = pd.read_csv(base_path + 'y_train_full.csv').values.ravel()

print(f"X_train_full shape (85% de los datos): {X_train_full.shape}")

### 2. Definición del Preprocesador (ColumnTransformer)

In [ ]:
vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = [
    'is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

### 3. Definimos los modelos que vamos a entrenar y el pipeline completo

In [ ]:
# --- 2. DEFINICIÓN DE MODELOS ---
RANDOM_STATE = 42

modelos = {
    "Baseline (mayoría)": DummyClassifier(strategy="most_frequent"),
    
    # --- Modelos CON class_weight='balanced' ---
    "Regresión Logística": LogisticRegression(C=1, max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
    "Árbol de Decisión": DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'),
    "Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    
    # --- Modelos SIN class_weight en sklearn ---
    "LDA": LinearDiscriminantAnalysis(),
    "QDA (regularizado)": QuadraticDiscriminantAnalysis(reg_param=0.3),
    "Naive Bayes": GaussianNB(),
    "k-NN (k=15)": KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    "Red Neuronal (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=RANDOM_STATE, early_stopping=True),
}

# --- 3. CONFIGURACIÓN DE CROSS-VALIDATION ---
# Usamos TimeSeriesSplit para respetar el orden temporal de las reservas
tscv = TimeSeriesSplit(n_splits=5)

# --- 4. FUNCIÓN DE EVALUACIÓN ---
def ejecutar_cv(pipeline, nombre_modelo):
    print(f"\n{'='*50}")
    print(f"Ejecutando CV para: {nombre_modelo}")
    print(f"{'='*50}")
    
    # Ejecutar validación cruzada
    cv_results = cross_validate(
        pipeline, 
        X_train_full,  # Asegúrate de que esta variable existe en tu notebook
        y_train_full,  # Asegúrate de que esta variable existe en tu notebook
        cv=tscv, 
        scoring=['roc_auc', 'f1', 'accuracy'],
        n_jobs=-1
    )
    
    # Mostrar resultados por cada fold temporal
    n_folds = len(cv_results['test_roc_auc'])
    for i in range(n_folds):
        print(f"Fold {i+1} - ROC-AUC: {cv_results['test_roc_auc'][i]:.4f} | F1: {cv_results['test_f1'][i]:.4f} | Accuracy: {cv_results['test_accuracy'][i]:.4f}")
    
    # Calcular promedios y desviaciones
    roc_auc_mean = cv_results['test_roc_auc'].mean()
    f1_mean = cv_results['test_f1'].mean()
    acc_mean = cv_results['test_accuracy'].mean()
    
    print("\n--- Resumen Global ---")
    print(f"ROC-AUC Promedio:  {roc_auc_mean:.4f} (+/- {cv_results['test_roc_auc'].std() * 2:.4f})")
    print(f"F1-Score Promedio: {f1_mean:.4f} (+/- {cv_results['test_f1'].std() * 2:.4f})")
    print(f"Accuracy Promedio: {acc_mean:.4f} (+/- {cv_results['test_accuracy'].std() * 2:.4f})")
    
    # Devolver métricas para la tabla final
    return {
        "Modelo": nombre_modelo,
        "ROC-AUC CV": roc_auc_mean,
        "F1-Score CV": f1_mean,
        "Accuracy CV": acc_mean
    }

# --- 5. BUCLE DE ENTRENAMIENTO Y EVALUACIÓN ---
resultados_cv = []

for nombre, modelo in modelos.items():
    # Crear el pipeline uniendo tu preprocesador y el modelo actual
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor), # Asegúrate de que 'preprocessor' ya está definido
        ('classifier', modelo)
    ])
    
    # Ejecutar la evaluación y guardar el resultado
    res = ejecutar_cv(pipeline, nombre)
    resultados_cv.append(res)

# --- 6. MOSTRAR TABLA COMPARATIVA FINAL ---
print("\n" + "="*70)
print("TABLA COMPARATIVA FINAL - CROSS VALIDATION (Ordenada por ROC-AUC)")
print("="*70)

# Crear DataFrame, ordenar por ROC-AUC y mostrar
df_resultados_cv = pd.DataFrame(resultados_cv)
df_resultados_cv = df_resultados_cv.sort_values(by="ROC-AUC CV", ascending=False).reset_index(drop=True)

# Usamos display() para que se vea bonito en Jupyter Notebook
display(df_resultados_cv) 


No estás viendo las matrices de confusión ni el reporte detallado con Precision y Recall por clase que veías en el script anterior.

Esto es completamente normal y ocurre por cómo funciona la función **`cross_validate`** de scikit-learn.

### ¿Por qué no aparecen?

1. **`cross_validate` solo devuelve números (scores):** 
   En el script anterior, entrenábamos el modelo una sola vez, hacíamos predicciones (`y_pred`) y con esas predicciones dibujábamos la matriz. En cambio, `cross_validate` entrena y evalúa el modelo **5 veces distintas** (por tus 5 folds temporales). Esta función está programada para devolver solo los promedios de las métricas que le pedimos (`roc_auc`, `f1`, `accuracy`), pero **no guarda las predicciones reales**. Al no tener las predicciones, no podemos dibujar la matriz.

2. **Tendrías demasiados gráficos:**
   Como el modelo se evalúa 5 veces en periodos de tiempo distintos, si imprimiéramos las matrices de confusión tendrías **5 matrices por cada modelo**. Con 9 modelos, ¡se imprimirían 45 matrices de confusión! Sería imposible de leer.

### ¿Qué se hace normalmente en este paso?

El flujo de trabajo correcto en Machine Learning es exactamente el que estás haciendo ahora:

1. **Fase de Selección (Donde estás ahora):** Usas Cross-Validation (solo con los números y la tabla final) para ver qué modelo es el más estable y robusto a lo largo del tiempo. Aquí no nos importan tanto los gráficos, solo queremos ver quién tiene el mejor `ROC-AUC` y `F1-Score` promedio.
2. **Fase de Análisis Profundo:** Una vez que la tabla final te dice (por ejemplo) que el *Gradient Boosting* y el *Random Forest* son los mejores, agarras **solo esos dos modelos**, los entrenas de forma tradicional (como en tu primer script) o usas una función llamada `cross_val_predict`, y ahí sí les sacas la matriz de confusión y el reporte detallado para analizarlos a fondo.

**En resumen:** El script funcionó perfecto. Esa salida en verde con los "Folds" y la tabla resumen al final es exactamente lo que necesitas en esta etapa para decidir con qué 2 o 3 modelos te vas a quedar para afinarles los hiperparámetros.